In [1]:
# Part 4 — LLM-Powered Feature
# Track chosen: (C) Model Prediction Explanation Pipeline

import os
import re
import json
import pandas as pd
import joblib
import requests
import jsonschema
from dotenv import load_dotenv

load_dotenv()

LLM_API_KEY = os.environ["OPENROUTER_API_KEY"]
LLM_API_URL = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "openrouter/auto"

In [2]:
best_model = joblib.load("best_model.pkl")

TRAIN_COLUMNS = [
    "longitude", "latitude", "housing_median_age", "total_rooms",
    "total_bedrooms", "population", "households", "median_income",
    "is_value_capped", "rooms_per_household", "bedrooms_per_room",
    "population_per_household", "income_category",
    "ocean_proximity_INLAND", "ocean_proximity_ISLAND",
    "ocean_proximity_NEAR BAY", "ocean_proximity_NEAR OCEAN",
]

INCOME_ORDER_MAP = {"Low": 0, "Medium": 1, "High": 2, "Very High": 3}

def encode_record(features):
    row = dict(features)
    row["income_category"] = INCOME_ORDER_MAP[row["income_category"]]
    ocean_value = row.pop("ocean_proximity")
    for col in TRAIN_COLUMNS:
        if col.startswith("ocean_proximity_"):
            row[col] = 1 if col == f"ocean_proximity_{ocean_value}" else 0
    return pd.DataFrame([row]).reindex(columns=TRAIN_COLUMNS, fill_value=0)

In [3]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

test_with_pii = "Contact the owner at jane.doe@example.com for details."
test_clean = "This record has no personal contact details, just housing statistics."

print("Test 1 (contains email) ->", "BLOCKED" if has_pii(test_with_pii) else "ALLOWED")
print("Test 2 (clean text)     ->", "BLOCKED" if has_pii(test_clean) else "ALLOWED")

Test 1 (contains email) -> BLOCKED
Test 2 (clean text)     -> ALLOWED


In [4]:
import time

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=800, max_retries=3):
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "Content-Type": "application/json",
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(LLM_API_URL, headers=headers, json=payload, timeout=30)
        except requests.exceptions.RequestException as e:
            print(f"Request failed (attempt {attempt}):", e)
            time.sleep(2 * attempt)
            continue

        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]

        if response.status_code == 429:
            print(f"Rate limited (attempt {attempt}), retrying after a short wait...")
            time.sleep(3 * attempt)
            continue

        print(f"LLM call failed with status code {response.status_code}: {response.text}")
        return None

    print("LLM call failed after all retry attempts.")
    return None

In [5]:
test_output = call_llm(
    "You are a helpful assistant. Reply with only one word.",
    "Reply with only the word: hello",
    temperature=0.0
)
print("Test call output:", test_output)

Test call output: hello


In [6]:
def get_class_and_probability(model, encoded_row):
    pred_class = model.predict(encoded_row)[0]
    proba_row = model.predict_proba(encoded_row)[0]
    class_index = list(model.classes_).index(pred_class)
    pred_proba = float(proba_row[class_index])
    return pred_class, pred_proba

In [7]:
SYSTEM_PROMPT = (
    "You are a structured model explanation generator. "
    "Return ONLY valid JSON with these exact keys: "
    "prediction_label, confidence_level, top_reason, second_reason, next_step. "
    "prediction_label must be a descriptive string such as 'expensive' or "
    "'not_expensive' — never a bare number or the raw predicted_class value. "
    "Do not include markdown, bullets, or extra text. "
    "Keep top_reason and second_reason to a single short sentence each. "
    "confidence_level must be one of low, medium, high."
)
USER_PROMPT_TEMPLATE = """Feature values:
{feature_json}

Predicted class:
{predicted_class}

Predicted probability:
{predicted_probability}

Return a JSON explanation using the required schema.
Example of a valid prediction_label value: "expensive" or "not_expensive" (not a number)."""

In [8]:
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step",
    ],
    "additionalProperties": False,
}

FALLBACK_EXPLANATION = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None,
}

In [9]:
def build_user_prompt(features, pred_class, pred_proba):
    return USER_PROMPT_TEMPLATE.format(
        feature_json=json.dumps(features, ensure_ascii=False),
        predicted_class=pred_class,
        predicted_probability=round(pred_proba, 4),
    )

In [10]:
def extract_json(text):
    """Strip markdown code fences and isolate the JSON object from the response."""
    text = text.strip()

    # Remove ```json ... ``` or ``` ... ``` wrappers if present
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()

    # Fall back to extracting the first {...} block if extra text remains
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    return text

In [11]:
def normalize_explanation(parsed, pred_class):
    """Coerce a numeric prediction_label into the expected descriptive string."""
    label = parsed.get("prediction_label")
    if isinstance(label, (int, float)):
        parsed["prediction_label"] = "expensive" if int(label) == 1 else "not_expensive"
    return parsed

def get_explanation(features, pred_class, pred_proba, temperature=0.0):
    user_prompt = build_user_prompt(features, pred_class, pred_proba)

    raw_response = call_llm(SYSTEM_PROMPT, user_prompt, temperature=temperature)

    if raw_response is None:
        return None, dict(FALLBACK_EXPLANATION), "fail (no response)"

    if not raw_response.strip():
        print("Empty response received from LLM.")
        return raw_response, dict(FALLBACK_EXPLANATION), "fail (empty response)"

    try:
        cleaned_response = extract_json(raw_response)
        parsed = json.loads(cleaned_response)
        parsed = normalize_explanation(parsed, pred_class)
    except json.JSONDecodeError as e:
        print("JSON decode error:", e)
        return raw_response, dict(FALLBACK_EXPLANATION), f"fail (invalid JSON: {e})"

    try:
        jsonschema.validate(instance=parsed, schema=EXPLANATION_SCHEMA)
    except jsonschema.ValidationError as e:
        print("Schema validation error:", e.message)
        return raw_response, dict(FALLBACK_EXPLANATION), f"fail (schema validation error: {e.message})"

    return raw_response, parsed, "pass"

In [12]:
sample_1 = {
    "longitude": -122.23, "latitude": 37.88, "housing_median_age": 41,
    "total_rooms": 880, "total_bedrooms": 129, "population": 322,
    "households": 126, "median_income": 8.3, "is_value_capped": 0,
    "ocean_proximity": "NEAR BAY", "income_category": "Very High",
    "rooms_per_household": 880 / 126, "bedrooms_per_room": 129 / 880,
    "population_per_household": 322 / 126,
}

sample_2 = {
    "longitude": -119.5, "latitude": 36.6, "housing_median_age": 25,
    "total_rooms": 3200, "total_bedrooms": 700, "population": 2100,
    "households": 650, "median_income": 3.1, "is_value_capped": 0,
    "ocean_proximity": "INLAND", "income_category": "Low",
    "rooms_per_household": 3200 / 650, "bedrooms_per_room": 700 / 3200,
    "population_per_household": 2100 / 650,
}

sample_3 = {
    "longitude": -118.4, "latitude": 34.1, "housing_median_age": 15,
    "total_rooms": 1500, "total_bedrooms": 300, "population": 900,
    "households": 400, "median_income": 5.2, "is_value_capped": 0,
    "ocean_proximity": "NEAR OCEAN", "income_category": "Medium",
    "rooms_per_household": 1500 / 400, "bedrooms_per_room": 300 / 1500,
    "population_per_household": 900 / 400,
}

In [13]:
sample_inputs = [sample_1, sample_2, sample_3]

prediction_records = []
for i, features in enumerate(sample_inputs, start=1):
    encoded = encode_record(features)
    pred_class, pred_proba = get_class_and_probability(best_model, encoded)
    prediction_records.append({
        "sample_id": f"sample_{i}",
        "features": features,
        "pred_class": pred_class,
        "pred_proba": pred_proba,
    })
    print(f"Sample {i}: predicted_class={pred_class}, probability={pred_proba:.4f}")

Sample 1: predicted_class=1, probability=0.9700
Sample 2: predicted_class=0, probability=0.9650
Sample 3: predicted_class=1, probability=0.8300


In [14]:
demo_rows = []
for rec in prediction_records:
    raw, parsed, status = get_explanation(
        rec["features"], rec["pred_class"], rec["pred_proba"], temperature=0.0
    )

    print("\n---", rec["sample_id"], "---")
    print("Input:", rec["features"])
    print("Raw LLM response:", raw)
    print("Validation status:", status)

    demo_rows.append({
        "Feature Input": rec["sample_id"],
        "Predicted Class": rec["pred_class"],
        "Probability": round(rec["pred_proba"], 4),
        "Explanation JSON": parsed,
        "Validation Status": status,
    })

demo_table = pd.DataFrame(demo_rows)
demo_table


--- sample_1 ---
Input: {'longitude': -122.23, 'latitude': 37.88, 'housing_median_age': 41, 'total_rooms': 880, 'total_bedrooms': 129, 'population': 322, 'households': 126, 'median_income': 8.3, 'is_value_capped': 0, 'ocean_proximity': 'NEAR BAY', 'income_category': 'Very High', 'rooms_per_household': 6.984126984126984, 'bedrooms_per_room': 0.14659090909090908, 'population_per_household': 2.5555555555555554}
Raw LLM response: ```json
{
  "prediction_label": "expensive",
  "confidence_level": "high",
  "top_reason": "The median income is very high, indicating a strong ability to afford higher housing prices.",
  "second_reason": "The location is 'NEAR BAY', which is often associated with more desirable and thus more expensive real estate.",
  "next_step": "Consider analyzing the specific features of properties in this area to understand the drivers of high housing costs."
}
```
Validation status: pass

--- sample_2 ---
Input: {'longitude': -119.5, 'latitude': 36.6, 'housing_median_age'

,Feature Input,Predicted Class,Probability,Explanation JSON,Validation Status
0,sample_1,1,0.970,"{'prediction_label': 'expensive', 'confidence_...",pass
1,sample_2,0,0.965,"{'prediction_label': 'not_expensive', 'confide...",pass
2,sample_3,1,0.830,"{'prediction_label': 'expensive', 'confidence_...",pass


In [15]:
temp_rows = []
for rec in prediction_records:
    _, parsed_0, _ = get_explanation(rec["features"], rec["pred_class"], rec["pred_proba"], temperature=0.0)
    _, parsed_7, _ = get_explanation(rec["features"], rec["pred_class"], rec["pred_proba"], temperature=0.7)

    temp_rows.append({
        "Input": rec["sample_id"],
        "Output at temp=0": parsed_0,
        "Output at temp=0.7": parsed_7,
        "Key difference": "temp=0 is deterministic; temp=0.7 may vary wording and rationale",
    })

temp_table = pd.DataFrame(temp_rows)
temp_table

,Input,Output at temp=0,Output at temp=0.7,Key difference
0,sample_1,"{'prediction_label': 'expensive', 'confidence_...","{'prediction_label': 'expensive', 'confidence_...",temp=0 is deterministic; temp=0.7 may vary wor...
1,sample_2,"{'prediction_label': 'not_expensive', 'confide...","{'prediction_label': 'expensive', 'confidence_...",temp=0 is deterministic; temp=0.7 may vary wor...
2,sample_3,"{'prediction_label': 'expensive', 'confidence_...","{'prediction_label': 'expensive', 'confidence_...",temp=0 is deterministic; temp=0.7 may vary wor...
